# L13a: Synthetic Markets with Jump Hidden Markov Models
In this lecture, we build synthetic markets: simulated price histories that share the features of real ones, such as fat tails and volatility clustering. A backtest tests a trading method on one historical path. How would the same method do on thousands of plausible paths, including sell-offs that never happened?

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Construct Markov chains and hidden Markov models:__ Define a state space, a transition matrix, and its stationary distribution, and describe how hidden states emit the values we observe.
> * __Build a jump hidden Markov model of one asset:__ Bin the daily growth rate into states, estimate the transition matrix by counting, fit an emission distribution in each state, and explain why jump episodes are needed to reproduce volatility clustering.
> * __Compose a synthetic market of many stocks:__ Combine a market draw, a sector draw, and a stock's own draw into synthetic stocks that move together like real ones.

Some results in this lecture are core, and some are here to build understanding:

> __What you need to know:__
>
> __Core concepts:__
> * A Markov chain is a set of states $\mathcal{S}$ and a transition matrix $\mathbf{P}$ whose rows sum to one. Its stationary distribution satisfies $\bar{\pi} = \bar{\pi}\mathbf{P}$.
> * The jump hidden Markov model bins the daily growth rate into states of equal probability, estimates $\mathbf{P}$ by counting, and draws each day's growth rate from a Student-t distribution of the day's state. Jump episodes, which start with probability $\epsilon$ and last $\lambda$ days on average, keep volatility clustering.
> * A synthetic stock is $g_{i} = \alpha_{i} + \beta_{i}\,g_{m} + \delta_{i}\,f_{s} + \psi_{i}\,\tilde{\varepsilon}_{i}$: the market and sector draws it shares with other stocks make them move together.
>
> __For understanding__ (you will not need to reproduce or memorize these):
> * The fitting and sampling algorithms of the jump hidden Markov model, step by step.
> * The tuning objective for $\epsilon$ and $\lambda$, and the shape of an episode compared with a real crash.
> * How the sector loadings and the own-draw scale are chosen, and the drifting $\beta$ option.

__Notation:__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L13a) lists this lecture's symbols with their meanings and units.

Let's get started!
___

## Examples
__We will work through these examples in lecture:__

> [▶ Build a jump hidden Markov model of `SPY`](CHEME-5660-L13a-Example-JumpHMM-SPY-Fall-2026.ipynb). In this example, we fit a hidden Markov model to the daily growth rate of `SPY` from 2014 to 2024, simulate it, and compare its distribution and volatility clustering with the data. We then tune jump episodes, see how much of the clustering they restore, and study what they change in the distribution.

> [▶ Build a synthetic market of many stocks](CHEME-5660-L13a-Example-JumpHMM-SectorFactor-Fall-2026.ipynb). In this example, we build a universe of two stocks from each sector plus `SPY` with the sector factor model, compare the correlations, volatilities, and β of a synthetic path with the real data, let β drift, and save the generator for later use.

___

## Review: Wolfram Markets
Previously, we built Wolfram markets, in which each trader follows a rule: a table from the sum of its four neighbors' intentions to its next intention. A small change in that table can turn a calm market into a crash.

> __Placeholder:__ panels of simulated Wolfram markets go here.

__Today:__ A market can change its character without warning. Our goal is a trade management system that makes money when it can and, at a minimum, preserves cash when it cannot.

To reach that goal, the system has to adapt as the market changes. To test such a system, we need many plausible paths of a changing market.
___

## Company Profile: Two Sigma
[Two Sigma](https://www.twosigma.com/) is a quantitative investment manager founded in 2001 by David Siegel and John Overdeck. The firm is based in New York and has about 1,700 employees. It says it follows "principles rooted in technology and data science as much as those found in financial services."

> __What are market regimes?__
>
> In October 2021, Two Sigma researchers Alex Botte and Doris Bao published [A Machine Learning Approach to Regime Modeling](https://www.twosigma.com/articles/a-machine-learning-approach-to-regime-modeling/). They let the data define the market regimes instead of labeling them by judgment.
>
> * __Four regimes from the data:__ The authors fit a Gaussian mixture model, a machine learning method that sorts data into clusters, to 17 drivers of returns, such as equity and inflation, from 1971 to 2020. It found Steady State (47% of the time), Inflation (20%), Crisis (17%), and Walking on Ice (16%), a regime in which bubbles may be forming.
> * __Regimes switch quickly:__ It was rare for one regime to last long without interruption from another. In 2020, Crisis appeared in February and March and was followed immediately by Walking on Ice. Steady State returned in the second half of the year.
> * __No memory:__ The model sorts each period on its own, with no awareness of the periods before or after it. The authors state that "this model is not predictive."

__Explore further:__

* __Jobs and internships:__ The [internships page](https://www.twosigma.com/careers/internships/) lists summer programs in quantitative research and software engineering at the New York office. In November 2026, the firm listed a [Quantitative Researcher Intern (Summer 2027)](https://careers.twosigma.com/careers/JobDetail/New-York-New-York-United-States-Quantitative-Researcher-Intern-2027-Summer/13945) role, a 10-week project with a mentor. The posting says, "You don't need a background in finance."
* __Video:__ Watch [Two Sigma Interns Share Their Experience](https://www.youtube.com/embed/xYqn7N1_xPs) on the firm's YouTube channel.
* __Recent work:__ In January 2026, Two Sigma leaders discussed where AI is heading in quantitative investing in [AI in Investment Management: 2026 Outlook](https://www.twosigma.com/articles/ai-in-investment-management-2026-outlook-part-i/) and [Part II](https://www.twosigma.com/articles/ai-in-investment-management-2026-outlook-part-ii/).

__Connection to today's lecture:__ Two Sigma's regimes play the role of our hidden states, the moods of the market. Their model has no memory, while our transition matrix gives the probability of moving from today's mood to tomorrow's.
___

## Why Synthetic Markets?
Previously, we tested trading methods by backtesting them on historical prices. A backtest is a single path: the market went through one sequence of booms and sell-offs, and a method that did well on that sequence may have been lucky. To judge a method, we would like to run it on many plausible paths, including bad ones that did not happen.

A __synthetic market__ is a model that generates such paths. To be useful for stress tests, its paths should share the features of real growth rates that matter for risk:
* __Fat tails:__ large daily moves are far more common than a normal distribution predicts.
* __Volatility clustering:__ large moves follow large moves, so the absolute growth rate stays correlated for weeks.
* __Co-movement:__ stocks move together, through the market and through their sectors.

Geometric Brownian motion, which we used before, can have co-movement through correlated shocks. It has neither fat tails nor volatility clustering, because its daily moves are normally distributed and its volatility is constant. In this lecture, we build a generator that has all three, starting from Markov chains.
___

## Markov Chains and Hidden Markov Models
A Markov chain is a stochastic (random) process that moves between states. The next state depends only on the present state. For example, the states could be the moods of an investor: bullish, neutral, or bearish.

> __Definition (Markov chain):__ A Markov chain has a set of states $\mathcal{S}=\left\{s_{1},\dotsc,s_{N}\right\}$ and a transition matrix $\mathbf{P}\in\mathbb{R}^{N\times{N}}$. Let $X_{t}$ be the state on day $t$. The element $p_{ij}\in\mathbf{P}$ is the probability of moving from $s_{i}$ today to $s_{j}$ tomorrow:
>
> $$
> p_{ij} = P\left(X_{t+1}=s_{j}\mid X_{t}=s_{i}\right)
> $$
>
> Each row of $\mathbf{P}$ sums to one and is a $\texttt{categorical}$ distribution over tomorrow's state. The __stationary distribution__ $\bar{\pi}$ solves $\bar{\pi} = \bar{\pi}\mathbf{P}$ and gives the long-run share of time the chain spends in each state.

We estimate the transition matrix $\mathbf{P}$ from data:

> __Proposition (estimating the transition matrix):__ In a sequence of observed states $X_{1},\dotsc,X_{T}$, let $n_{ij}$ count the days in state $s_{i}$ followed by a day in state $s_{j}$. The maximum likelihood estimate of $p_{ij}$ is the share of the days in state $s_{i}$ that moved to state $s_{j}$:
>
> $$
> \boxed{\hat{p}_{ij} = \frac{n_{ij}}{\sum_{k=1}^{N} n_{ik}}}
> $$
>
> This estimate maximizes the log-likelihood $\sum_{i,j} n_{ij}\ln p_{ij}$, subject to each row of $\mathbf{P}$ summing to one.

A __hidden Markov model__ adds observable outputs to the chain states. However, the states $X_{t}$ of the chain are hidden. Each day, we observe an output (emission) drawn from a distribution that depends on that day's hidden state.

For a market generator, the hidden state sets how volatile the market is today, and the emission turns the state into a growth rate. A generator draws the first state of each path from $\bar{\pi}$. The [Markov models notebook](advanced/markov/CHEME-5660-L13a-Lecture-MarkovModels-Fall-2026.ipynb) covers the Markov property, power iteration for $\bar{\pi}$, and sampling a hidden Markov model.
___

## The Jump Hidden Markov Model
Let's build a hidden Markov model of the daily growth rate $g_{t} = \frac{1}{\Delta t}\ln\left(S_{t}/S_{t-1}\right)$ (units: 1/yr) of one asset, following Alswaidan and Varner, who call it a jump hidden Markov model. It has three parts:
* __States:__ We fit a Laplace distribution to the growth rates and use its cumulative distribution function $F_{L}$ to cut it into $N$ slices of equal probability. Each slice is a hidden state, from state 1 with the most negative growth rates to state $N$ with the most positive. In the data, we see each day's state, but in simulation the states are hidden, and only their growth rates are reported.
* __Transition matrix:__ Each day of the data falls in one slice, which gives a sequence of states. We estimate $\hat{\mathbf{P}}$ by counting the moves between the states of consecutive days.
* __Emissions:__ In each state $s_{k}$, the growth rate is drawn from a Student-t distribution, $g_{t} \mid X_{t} = s_{k} \sim \mu_{k} + \sigma_{k}\,T_{\nu}$. The mean $\mu_{k}$ and the standard deviation $\sigma_{k}$ come from the growth rates of the days in state $s_{k}$, and $T_{\nu}$ has $\nu$ degrees of freedom. With $\nu = 5$, the draws in a state spread about 1.3 times $\sigma_{k}$.

The schematic below shows the fit with $N = 5$ states. Each state is a mood of the market, from very bearish (state 1) to very bullish (state 5):

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13a-JumpHMM-Fit-Schematic.svg" width="1000" style="max-width:100%; height:auto;" alt="The fit of the model in four labeled steps. On the left, a plot of the daily growth rate of a ticker on sixteen trading days, with five horizontal bands shaded behind the points. Each point is a small circle with the number of its state inside: 3, 4, 3, 2, 3, 4, 5, 1, 5, 1, 2, 3, 4, 3, 3, 2. The three moves from state 3 to state 4 on consecutive days are orange segments of the line, and a key in the corner reads a move 3 to 4. In the middle, under the heading 1. Laplace, the fitted Laplace distribution is drawn sideways on the same growth-rate axis, with its peak in the middle band. The band boundaries cut the area under the distribution into N = 5 slices of equal probability, each labeled 20 percent. A bar to the right, headed 2. States, names the state and the mood of each band: state 1, very bearish, in red at the bottom, then state 2, bearish, state 3, neutral, state 4, bullish, and state 5, very bullish, in blue at the top. An orange arrow labeled n i j leads from the bar to a table of counts headed 3. Count. The table has the state on day t in the rows and the state on day t plus 1 in the columns, and the cell in row 3 and column 4 holds 3 and is highlighted in orange. An arrow leads from the table to a box headed 4. Transitions, with the estimate of p i j, n i j divided by the sum of row i, and the example for row 3, 3 over 6."/>
    </center>
</div>

### Fitting algorithm
The algorithm below fits the model to the daily growth rates of one ticker.

__Initialize__: Given the daily growth rates $g_{1},\dotsc,g_{T}$ of one ticker, the number of states $N$, and the degrees of freedom $\nu$ of the emissions. Set the counts $n_{ij}\gets 0$ for every pair of states $s_{i}$ and $s_{j}$.

1. __Laplace:__ Fit the Laplace distribution by maximum likelihood. Set the location $\mu$ to the median of the growth rates, and set the scale $b \gets \frac{1}{T}\sum_{t=1}^{T}\lvert g_{t} - \mu\rvert$.
2. __States:__ For each day $t = 1,2,\dots,T$ __do__:
    - a. Compute the cumulative probability $F_{L}(g_{t})$.
    - b. Find the slice that holds it: set $X_{t}\gets s_{k}$, where $(k-1)/N \leq F_{L}(g_{t}) < k/N$.
3. __Count:__ For each day $t = 2,3,\dots,T$ __do__:
    - a. Find the states of the two days: $X_{t-1} = s_{i}$ and $X_{t} = s_{j}$.
    - b. Count the move: $n_{ij}\gets n_{ij} + 1$.
4. __Transitions:__ Divide each row of counts by its sum: $\hat{p}_{ij}\gets n_{ij}/\sum_{k=1}^{N} n_{ik}$ for every pair of states $s_{i}$ and $s_{j}$.
5. __Emissions:__ For each state $s_{k}$ __do__:
    - a. Collect the growth rates of the days with $X_{t} = s_{k}$.
    - b. Set $\mu_{k}$ to their mean and $\sigma_{k}$ to their standard deviation.
6. __Start:__ Compute the stationary distribution $\bar{\pi}$ of $\hat{\mathbf{P}}$.

__Output__: The Laplace location $\mu$ and scale $b$, the transition matrix $\hat{\mathbf{P}}$, the emission parameters $\mu_{k}$ and $\sigma_{k}$ of each state, and the stationary distribution $\bar{\pi}$.

__Notes:__
1. The Laplace cumulative distribution function has a closed form: $F_{L}(g) = \frac{1}{2}\exp\left((g-\mu)/b\right)$ for $g < \mu$, and $F_{L}(g) = 1 - \frac{1}{2}\exp\left(-(g-\mu)/b\right)$ for $g \geq \mu$.
2. Step 2 gives the same states as the slice edges $F_{L}^{-1}(k/N)$: a growth rate between two neighboring edges has a cumulative probability between $(k-1)/N$ and $k/N$.
3. `JumpHMM.jl` handles two rare cases. A state with no counted moves out of it gets the uniform row $\hat{p}_{ij} = 1/N$. A state with fewer than two days uses the mean and the standard deviation of all the growth rates.

The Student-t emissions and the wide extreme slices give the generator fat tails. To simulate a path, we draw a starting state from $\bar{\pi}$, step the chain with $\hat{\mathbf{P}}$, and draw each day's growth rate from the emission of the day's state.

### Why add jumps?
The counted chain forgets quickly. After a volatile day, the next few transitions usually take the chain from an extreme state back to the middle states. The absolute growth rate of a simulated path stays correlated for only a few days, while in real markets volatile periods last for weeks.

The jump hidden Markov model adds __jump episodes__ to keep volatile periods going. On a day with no episode under way, an episode starts with probability $\epsilon$ and lasts a Poisson number of days with mean $\lambda$. Each episode day draws its state from the most negative states with probability $p_{\text{neg}}$, slightly above one half, and from the most positive states otherwise.

An episode is a run of large moves of both signs, not a single price jump. The schematic below puts the pieces together for one simulated day:

<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13a-JumpHMM-Schematic.svg" width="880" style="max-width:100%; height:auto;" alt="A row of hidden states numbered 1, 2, through k, through N minus 1 and N, ordered from the most negative to the most positive daily growth rate, with double-headed arrows between neighbors. States 1 and 2 are red and braced as the most negative states S minus, and states N minus 1 and N are blue and braced as the most positive states S plus. A bell curve above each state is its emission, and state k emits mu k plus sigma k times a Student-t variable. Under state k, a note reads ordinary day: step with the estimated transition matrix. A dashed arrow from state k, marked no episode under way, leads up to a box: jump episode, starts with probability epsilon, lasts a Poisson number of days with mean lambda. From the box, a red arrow runs to the most negative states, taken on each episode day with probability p neg, and a blue arrow runs to the most positive states, taken with probability 1 minus p neg."/>
    </center>
</div>

### Sampling algorithm
The algorithm below draws one path of $T$ days from the fitted model with jump episodes.

__Initialize__: Given the transition matrix $\hat{\mathbf{P}}$, the emission parameters $\mu_{k}$ and $\sigma_{k}$ of each state, the degrees of freedom $\nu$, the stationary distribution $\bar{\pi}$, the episode probability $\epsilon$, the mean episode length $\lambda$, the number of tail states $N_{\text{tail}}$, the probability $p_{\text{neg}}$ that an episode day is negative, and the number of days $T$. The most negative states are $\mathcal{S}_{-} = \left\{s_{1},\dotsc,s_{N_{\text{tail}}}\right\}$, and the most positive states are $\mathcal{S}_{+} = \left\{s_{N-N_{\text{tail}}+1},\dotsc,s_{N}\right\}$. Draw the first state $X_{1}\sim\bar{\pi}$ and its growth rate $g_{1}$ from the emission of $X_{1}$. Set the number of episode days left $\texttt{remaining}\gets 0$.

For each day $t = 2,3,\dots,T$ __do__:
1. __New episode:__ If $\texttt{remaining} = 0$, __then__ draw $u\sim\text{Uniform}(0,1)$. If $u < \epsilon$, __then__ an episode starts: draw its length $\texttt{remaining}\sim\text{Poisson}(\lambda)$.
2. __State:__ Choose the state of day $t$:
    - If $\texttt{remaining} > 0$, __then__ day $t$ is an episode day. Draw $v\sim\text{Uniform}(0,1)$. If $v < p_{\text{neg}}$, __then__ draw $X_{t}$ uniformly from $\mathcal{S}_{-}$. Otherwise, draw $X_{t}$ uniformly from $\mathcal{S}_{+}$. Set $\texttt{remaining}\gets\texttt{remaining} - 1$.
    - If $\texttt{remaining} = 0$, __then__ day $t$ is an ordinary day. Draw $X_{t}$ from the $\texttt{categorical}$ distribution defined by row $X_{t-1}$ of $\hat{\mathbf{P}}$.
3. __Emission:__ Draw the growth rate from the emission of the day's state: $g_{t}\gets\mu_{k} + \sigma_{k}\,T_{\nu}$, where $X_{t} = s_{k}$ and $T_{\nu}$ is a new Student-t draw.
4. __Store:__ Store $X_{t}$ and $g_{t}$.

__Output__: The states $\{X_{1},\dotsc,X_{T}\}$ and the growth rates $\{g_{1},\dotsc,g_{T}\}$ of one path.

__Notes:__
1. With $\epsilon = 0$ no episode starts, and the algorithm samples a hidden Markov model with no jumps.
2. A Poisson draw can be zero. In that case no episode starts, and day $t$ is an ordinary day.
3. The check in step 1 runs again on the day after an episode ends, so a new episode can follow at once.
4. `JumpHMM.jl` sets $N_{\text{tail}} = 5$ and $p_{\text{neg}} = 0.52$ unless we choose other values.


### Tuning and checking the model
We choose $\epsilon$ and $\lambda$ by a grid search. For each pair, we simulate paths and keep the $M$ paths that contain an episode. The score $J$ measures how far each path's autocorrelation of $|g_{t}|$ at lags 1 to 25 and its kurtosis are from the data's, averaged over the paths:
$$
J(\epsilon,\lambda) = \frac{1}{M}\sum_{m=1}^{M}\left[\sum_{k=1}^{25}\left(\rho_{k} - \hat{\rho}_{m,k}\right)^{2} + 0.2\left(\kappa - \hat{\kappa}_{m}\right)^{2}\right]
$$
where $\rho_{k}$ and $\kappa$ are the autocorrelation of $|g_{t}|$ at lag $k$ and the kurtosis of the data, and $\hat{\rho}_{m,k}$ and $\hat{\kappa}_{m}$ are those of simulated path $m$.

We keep the pair with the smallest $J$. Because the score uses a finite number of simulated paths, the chosen pair can change with the random seed. Finally, we check the tuned model against the data: the distribution of each simulated path (a Kolmogorov–Smirnov test), its kurtosis, and its volatility clustering.

### What the episodes add and what they miss
Jump episodes add volatility clustering, but they also change the distribution of the growth rate. More or longer episodes make large moves common. The simulated tails then thin out, and the spread of the growth rate rises above the real one.

The episodes also have a different shape from a real crash. Every episode day draws its state from the same tail states. The volatility of an episode switches on, stays at one level for about $\lambda$ days, and switches off.

In 2020, the volatility of `SPY` rose to several times its usual level within a month. It then faded over the next few months.

The figure below compares the two shapes. The left panel is a schematic. The right panel shows the standard deviation of $g_{t}$ over the past 21 trading days, for `SPY` from February 20, 2020, and for the episodes of the tuned model (six examples and the mean over all episodes), each aligned at its first day:

<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13a-Episode-Shape.svg" width="1000" style="max-width:100%; height:auto;" alt="Two panels with volatility on the vertical axis and trading days since the start on the horizontal axis. The left panel, a schematic: a dashed gray line marks the volatility of ordinary days. A red box, filled in light red, rises from the ordinary level to a middle level at the start, stays flat, and drops back after a red double arrow labeled about lambda days; it is labeled simulated episode, flat for about lambda days. A thick black curve labeled real crash (SPY, 2020), spikes, then fades, rises to a peak about twice as high as the box within the first few weeks, falls below the box level well before the box ends, and then fades slowly. The right panel, the data: the standard deviation of g over the past 21 trading days, in 1/yr, from 30 days before the start to 200 days after it. The black curve, real SPY from February 20, 2020, rises from about 2 to about 15 near day 30 and falls back to about 4 by day 60. The thick red curve, the mean over the episodes of the tuned model, rises to about 7 within a month, stays near 7 until about day 100, and falls to about 2.6 by day 150. Six thin red curves, single episodes, have the same flat shape at levels between about 5 and 10. A dashed line marks day 0."/>
    </center>
</div>

A stress test on these paths meets long, moderate volatile periods rather than sharp crashes that fade.

> __Example:__
>
> [▶ Build a jump hidden Markov model of `SPY`](CHEME-5660-L13a-Example-JumpHMM-SPY-Fall-2026.ipynb). In this example, we fit a hidden Markov model to the daily growth rate of `SPY` from 2014 to 2024, simulate it, and compare its distribution and volatility clustering with the data. We then tune jump episodes, see how much of the clustering they restore, and study what they change in the distribution.

Next, let's extend the generator from one asset to many.
___

## Many Assets: The Sector Factor Model
Previously, we studied __factor models__, which explain how stocks move together with a few common factors.

For stock $i$ on day $t$, a factor model is given by:
$$
g_{i}(t) = \underbrace{\alpha_{i}}_{\text{firm}} + \underbrace{\beta_{i}\,g_{m}(t)}_{\text{market}} + \underbrace{\sum_{j}\delta_{ij}\,f_{j}(t)}_{\text{other factors}} + \underbrace{\varepsilon_{i}(t)}_{\text{residual}}
$$
where $g_{m}(t)$ is the growth rate of the market, the $f_{j}(t)$ are other factors, such as interest rates or inflation, and $\beta_{i}$ and $\delta_{ij}$ are the stock's sensitivities to the factors.

The __single index model__ keeps only the market factor and assumes that the residuals of different stocks are uncorrelated. In the data, the residuals of stocks in the same sector, such as two banks, are correlated. A stress test of a portfolio needs synthetic stocks that move together the way real ones do.

> __Definition (sector factor model):__ Each stock has one other factor besides the market, the factor $f_{s}(t)$ of its sector $s$.
>
> The daily growth rate of stock $i$ in sector $s$ is given by:
>
> $$
> g_{i}(t) = \alpha_{i} + \beta_{i}\,\underbrace{g_{m}(t)}_{\text{market}} + \delta_{i}\,\underbrace{f_{s}(t)}_{\text{sector}} + \psi_{i}\,\underbrace{\tilde{\varepsilon}_{i}(t)}_{\text{own}}
> $$
>
> where each draw comes from a jump hidden Markov model:
> * __Market:__ The market draw $g_{m}(t)$ comes from a model of `SPY`. The coefficients $\alpha_{i}$ and $\beta_{i}$ are the stock's single index model estimates, which we hold fixed.
> * __Sector:__ The sector draw $f_{s}(t)$ comes from a model of the average single index residual of the stocks in sector $s$. The loading $\delta_{i}$ is the slope of the stock's residual on the other residuals in its sector, scaled to match the sector's co-movement in the data.
> * __Own:__ The own draw $\tilde{\varepsilon}_{i}(t)$ comes from a model of what is left of the stock's single index residual $\varepsilon_{i}(t)$ after the sector term, $\varepsilon_{i}(t)-\delta_{i}f_{s}(t)$, centered on the model's long-run mean. The scale $\psi_{i}$ sizes the own draw, so that the sector and own terms together keep the variance of $\varepsilon_{i}(t)$.
>
> On each day, every stock shares the market draw, and every stock in a sector shares the sector draw. These shared draws are the only link between stocks.

### Sampling Algorithm
The algorithm below draws one synthetic path of the whole universe over $T$ days.

__Initialize__: Given the jump hidden Markov models of the market, of each sector's factor, and of each stock's residual after its sector term; each stock's single index estimates $\alpha_{i}$ and $\beta_{i}$ and sector loading $\delta_{i}$; the scale of each stock's own draw, $\psi_{i}^{2} = \left(\operatorname{Var}(\varepsilon_{i}) - \delta_{i}^{2}\operatorname{Var}(f_{s})\right)/\operatorname{Var}(\varepsilon_{i}-\delta_{i}f_{s})$, with the variances of the real data; each stock's real starting price $S_{i}(0)$; and the number of days $T$. Draw a market path $g_{m}(1),\dotsc,g_{m}(T)$ from the market model. For each sector $s$, draw a factor path $f_{s}(1),\dotsc,f_{s}(T)$ from the model of sector $s$, independently of the other sectors and of the market.

For each stock $i$ in sector $s$ __do__:
1. __Own draw:__ Draw a path of the stock's own model and subtract the model's long-run mean (the mean of one long simulated path) to get $\tilde{\varepsilon}_{i}(1),\dotsc,\tilde{\varepsilon}_{i}(T)$.
2. __Compose:__ For each day $t = 1,2,\dots,T$ __do__:
    - a. Compose the growth rate: $g_{i}(t) \gets \alpha_{i} + \beta_{i}\,g_{m}(t) + \delta_{i}\,f_{s}(t) + \psi_{i}\,\tilde{\varepsilon}_{i}(t)$.
    - b. Compound the price, starting from $S_{i}(0)$: $S_{i}(t) \gets S_{i}(t-1)\exp\left(g_{i}(t)\,\Delta t\right)$.

__Output__: The market path $g_{m}(t)$, and the growth rates $g_{i}(t)$ and prices $S_{i}(t)$ of every synthetic stock, for $t = 1,\dotsc,T$.

__Notes:__
1. The sector factor averages the residuals of the stocks in sector $s$, stock $i$ included. Independent draws drop the part of the stock's leftover residual that moves with the sector factor, and $\psi_{i}$ puts that part back. In a sector with many stocks, $\psi_{i}$ is close to one.
2. With the drifting β option below, $\beta_{i}$ becomes $\beta_{i}(t)$ in step 2.

### What the model keeps and what it gives up
Every draw comes from a jump hidden Markov model, so each synthetic stock has fat tails and clustered volatility. Its tails and clustering are those of the sum of its three draws, rather than an exact copy of the real stock's.

The stocks in a sector move together through their shared sector draw, and all stocks move together through the market draw. The sector draws are independent of each other. Two stocks in different sectors are therefore linked only through the market.

__Why not a copula?__ A copula imposes a fitted correlation by reordering each stock's draws across days. The reordering would scramble the timing of the volatile periods and jump episodes in those draws. Composing stocks from shared draws on the same day leaves that timing as it was simulated.

### An option: a drifting β
Real β wanders over time, but the model above holds $\beta_{i}$ fixed. The model can instead let $\beta_{i}(t)$ drift slowly around $\hat{\beta}_{i}$ as an exponential moving average of random shocks. The shocks are sized by how much the online estimate of β wanders in the real data.

A trading method that re-estimates β as it trades then faces the same problem in the synthetic market as in the real one.

> __Example:__
>
> [▶ Build a synthetic market of many stocks](CHEME-5660-L13a-Example-JumpHMM-SectorFactor-Fall-2026.ipynb). In this example, we build a universe of two stocks from each sector plus `SPY` with the sector factor model, compare the correlations, volatilities, and β of a synthetic path with the real data, let β drift, and save the generator for later use.
___

## Optional Advanced Material
> [▶ Review Markov chains and hidden Markov models](advanced/markov/CHEME-5660-L13a-Lecture-MarkovModels-Fall-2026.ipynb). In this notebook, we define the Markov property and the transition matrix, compute the stationary distribution by power iteration, and sample a hidden Markov model. Its two examples apply a hidden Markov model to a person's mood and to the daily growth rate of `SPY`.
___

## Summary
This lecture builds synthetic markets for stress tests, from Markov chains and hidden Markov models to a jump hidden Markov model of one asset and a sector factor model of many stocks.

> __Key Takeaways:__
>
> * __Markov chains and hidden Markov models describe sequences with memory of the present state:__ We defined the state space, the transition matrix, and the stationary distribution, and we estimated the transition matrix by counting transitions. Hidden states extend the chain, and their emission distributions produce the values we observe.
> * __The jump hidden Markov model reproduces fat tails and part of volatility clustering:__ We binned the daily growth rate into states of equal probability, counted transitions, and drew growth rates from a Student-t distribution in each state. Because the counted chain forgets volatile days quickly, we added jump episodes and tuned their probability and length against the data. Episodes add clustering, but they thin the tails and stay flat where a real crash spikes and fades.
> * __Shared draws make synthetic stocks move together:__ We composed each synthetic stock from a market draw and a sector draw that it shares with other stocks, plus a draw of its own residual. We sized the own draw so that the sector and own terms keep the variance of the stock's single index residual.

Synthetic markets let us test a trading method on many plausible paths, including sell-offs that never happened, rather than on one historical path.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___